### Rag Pipe line - Data ingestion to vector db pipeline

In [33]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [34]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: mcp.pdf
  ✓ Loaded 19 pages

Processing: Rudra Tiwari Brightly India Offer Letter.pdf
  ✓ Loaded 9 pages

Processing: rudra_tiwariii.pdf
  ✓ Loaded 1 pages

Total documents loaded: 29


In [35]:
all_pdf_documents

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-10-22T13:34:32+00:00', 'source': '..\\data\\pdf\\mcp.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1', 'source_file': 'mcp.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-10-22T13:34:32+00:00', 'source': '..\\data\\pdf\\mcp.pdf', 'total_pages': 19, 'page': 1, 'page_label': '2', 'source_file': 'mcp.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-10-22T13:34:32+00:00', 'source': '..\\data\\pdf\\mcp.pdf', 'total_pages': 19, 'page': 2, 'page_label': '3', 'source_file': 'mcp.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-10-22T13:34:32+00:00', 'source': '..\\data\\pdf\\mcp.pdf', 'total_pages': 19,

In [36]:
### Text splitting get into chunks

def split_documents (documents, chunk_size=1000,chunk_overlap=200):
            """Split documents into smaller chunks for better RAG performance"""
            text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
            )
            split_docs = text_splitter.split_documents (documents)
            print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
            # Show example of a chunk
            if split_docs:
                print(f"\nExample chunk:")
                print (f"Content: {split_docs[0].page_content[:200]}...")
                print(f"Metadata: {split_docs[0].metadata}")
            return split_docs




In [37]:
chunks = split_documents(all_pdf_documents)
chunks

Split 29 documents into 35 chunks

Example chunk:
Content: Confidential  Brightly Software India Private Ltd. 
 
 
 
8th Floor, BPTP Capital City  info@brightlysoftware.com 
Sector-94, Noida-201301  www.brigthlysoftware.com 
Uttar Pradesh, India 
 
Page 1 of ...
Metadata: {'producer': 'PDFKit.NET 12.3.563.0 DMV10', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-09-09T17:53:55+05:30', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_enabled': 'true', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_method': 'Standard', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_siteid': '38ae3bcd-9579-4fd4-adda-b42e1495d55a', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_actionid': 'eb8922b5-3512-490e-b73a-acad35181dee', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_contentbits': '0', 'author': 'Jhaveri, Deesha (LC RC-IN C)', 'moddate': '2025-09-17T02:39:35-07:00', 'source': '..\\data\\pdf\\Rudra Tiwari Brightly India Offer Letter.pdf', 'total_pages': 9, 'page': 0, 'pa

[Document(metadata={'producer': 'PDFKit.NET 12.3.563.0 DMV10', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-09-09T17:53:55+05:30', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_enabled': 'true', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_method': 'Standard', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_siteid': '38ae3bcd-9579-4fd4-adda-b42e1495d55a', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_actionid': 'eb8922b5-3512-490e-b73a-acad35181dee', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_contentbits': '0', 'author': 'Jhaveri, Deesha (LC RC-IN C)', 'moddate': '2025-09-17T02:39:35-07:00', 'source': '..\\data\\pdf\\Rudra Tiwari Brightly India Offer Letter.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'Rudra Tiwari Brightly India Offer Letter.pdf', 'file_type': 'pdf'}, page_content="Confidential  Brightly Software India Private Ltd. \n \n \n \n8th Floor, BPTP Capital City  info@brightlysoftware.com \nSector-94, Noida-201

### Embedding and vector store db

In [38]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [39]:

class EmbeddingManager:

    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name: str ="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name=:Hugging face model name for sentence embeddings
        """

        self.model_name=model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
        NumPy array of embeddings with shape
        (len(texts), embedding_dim).
        """
        if not self.model:
             raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
             texts,
             show_progress_bar=True,
             convert_to_numpy=True
            )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1609.09it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\rudra\AppData\Local\Temp\ipykernel_9628\3369941727.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store


In [40]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store it store in hard disk
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            # create a client that take the reference of vector store of chroma db
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 70


In [41]:
chunks

[Document(metadata={'producer': 'PDFKit.NET 12.3.563.0 DMV10', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-09-09T17:53:55+05:30', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_enabled': 'true', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_method': 'Standard', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_siteid': '38ae3bcd-9579-4fd4-adda-b42e1495d55a', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_actionid': 'eb8922b5-3512-490e-b73a-acad35181dee', 'msip_label_9d258917-277f-42cd-a3cd-14c4e9ee58bc_contentbits': '0', 'author': 'Jhaveri, Deesha (LC RC-IN C)', 'moddate': '2025-09-17T02:39:35-07:00', 'source': '..\\data\\pdf\\Rudra Tiwari Brightly India Offer Letter.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'Rudra Tiwari Brightly India Offer Letter.pdf', 'file_type': 'pdf'}, page_content="Confidential  Brightly Software India Private Ltd. \n \n \n \n8th Floor, BPTP Capital City  info@brightlysoftware.com \nSector-94, Noida-201

In [42]:
from openai.types import vector_store
### convert the test to embeddings
texts = [doc.page_content for doc in chunks]

## Generate the embeddings

embeddings = embedding_manager.generate_embeddings(texts)

##store in vector store

vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 35 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.50it/s]


Generated embeddings with shape: (35, 384)
Adding 35 documents to vector store...
Successfully added 35 documents to vector store
Total documents in collection: 105


# Retriver pipeline

In [43]:

class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(self, vector_store, embedding_manager):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query.
            top_k: Number of top results to return.
            score_threshold: Minimum similarity score threshold.

        Returns:
            List of dictionaries containing retrieved documents and metadata.
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        if not query.strip():
            return []

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        try:
            # Search in ChromaDB
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            print(f"The result is : {results}")

            retrieved_docs = []
            
            # Check documents in results
            if results.get('documents') and len(results['documents']) > 0 and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0] if results.get('metadatas') else [{}] * len(documents)
                distances = results['distances'][0] if results.get('distances') else [0.0] * len(documents)
                ids = results['ids'][0] if results.get('ids') else [f"doc_{i}" for i in range(len(documents))]

                # Process retrieved results
                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # For normalized embeddings in ChromaDB:
                    # Default L2 distance space: similarity = 1 - (distance / 2)
                    # Cosine distance space: similarity = 1 - distance
                    similarity_score = 1.0 - (distance / 2.0) if distance <= 2.0 else 1.0 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata or {},
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

            print(
                f"Retrieved {len(retrieved_docs)} documents "
                f"(after filtering)"
            )

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Initialize the retriever
rag_retriever = RAGRetriever(
    vector_store=vectorstore,
    embedding_manager=embedding_manager
)

In [44]:
rag_retriever.retrieve("What is skills of rudra tiwari?")

Retrieving documents for query: 'What is skills of rudra tiwari?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.90it/s]

Generated embeddings with shape: (1, 384)
The result is : {'ids': [['doc_94b17552_30', 'doc_cc6efd71_30', 'doc_ce1bb1c0_30', 'doc_f4314d70_6', 'doc_d080c64f_6']], 'embeddings': None, 'documents': [['Rudra Tiwari\n+91 7355402136|rudra.tiwari.4141@gmail.com|LinkedIn|GitHub\nProfessional Summary\nSoftware Engineer specializing inJava, Spring Boot, REST APIs, and microservices, with hands-on experience building\nauthentication, onboarding, loyalty, observability, andAI-enabled log investigation systems. Strong foundation inDSA, system\ndesign, databases, distributed systems, and Generative AI, with experience developing production-grade backend solutions.\nTechnical Skills\nLanguages:Java, Python, SQL, JavaScript, TypeScript\nBackend:Spring Boot, Spring AI, FastAPI, Microservices, Kafka, Node.js\nAI & GenAI:LLMs, Spring AI, Prompt Engineering, AI Agents\nDatabases:MySQL, PostgreSQL, MongoDB, Redis\nCloud & DevOps:A WS, Azure, Docker, Git, CI/CD\nObservability:OpenTelemetry, Distributed Tra

[{'id': 'doc_94b17552_30',
  'content': 'Rudra Tiwari\n+91 7355402136|rudra.tiwari.4141@gmail.com|LinkedIn|GitHub\nProfessional Summary\nSoftware Engineer specializing inJava, Spring Boot, REST APIs, and microservices, with hands-on experience building\nauthentication, onboarding, loyalty, observability, andAI-enabled log investigation systems. Strong foundation inDSA, system\ndesign, databases, distributed systems, and Generative AI, with experience developing production-grade backend solutions.\nTechnical Skills\nLanguages:Java, Python, SQL, JavaScript, TypeScript\nBackend:Spring Boot, Spring AI, FastAPI, Microservices, Kafka, Node.js\nAI & GenAI:LLMs, Spring AI, Prompt Engineering, AI Agents\nDatabases:MySQL, PostgreSQL, MongoDB, Redis\nCloud & DevOps:A WS, Azure, Docker, Git, CI/CD\nObservability:OpenTelemetry, Distributed Tracing, Structured Logging\nDeveloper T ools:Claude Code, Cursor, GitHub Copilot\nWork Experience\nSoftware Engineer Oct 2025 – Sept 2026\nSalescode AI Gurugram